In [0]:
from pyspark.sql import functions as F

bronze = spark.table("bronze.retail_sales_raw")

silver = (
    bronze
    .dropDuplicates(["transaction_id"])
    .filter(F.col("cost").isNotNull() & F.col("price").isNotNull())
    .withColumn("cost", F.col("cost").cast("double"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("transactional_date", F.to_timestamp("transactional_date", "dd-MM-yyyy HH:mm"))
    .withColumn("loyalty_card", F.when(F.col("loyalty_card") == "T", True).otherwise(False))
    .withColumn("revenue", F.col("price") * F.col("quantity"))
    .withColumn("margin", (F.col("price") - F.col("cost")) * F.col("quantity"))
)

rejects = bronze.filter(F.col("cost").isNull() | F.col("price").isNull())

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
silver.write.format("delta").mode("overwrite").saveAsTable("silver.retail_sales_clean")
rejects.write.format("delta").mode("overwrite").saveAsTable("silver.retail_sales_rejects")

print(f"Silver rows: {silver.count()}, Rejected rows: {rejects.count()}")

Silver rows: 4410, Rejected rows: 2833


In [0]:
display(silver.select("transaction_id", "transactional_date", "revenue", "margin").limit(5))

transaction_id,transactional_date,revenue,margin
3193.0,2022-01-21T16:24:00.000Z,52.769999999999996,5.4300000000000015
3212.0,2022-01-23T14:54:00.000Z,44.37,3.6299999999999972
3215.0,2022-01-23T17:22:00.000Z,8.79,0.7099999999999991
3242.0,2022-01-25T20:56:00.000Z,13.18,2.08
3259.0,2022-01-26T23:59:00.000Z,1.09,1.0
